# GBM Options Pricing, Forecasting & an RL Agent That Buys and Sells Calls and Puts

This notebook builds an options model on **geometric Brownian motion**:

1. **Volatility forecasting.** HAR-RV (Corsi), GARCH(1,1), EWMA and historical volatility are compared out-of-sample against implied volatility (VIX). The literature shows HAR is a strong baseline, and that volatility is "rough" (Gatheral et al.).
2. **GBM pricing engine.** Monte Carlo is verified against Black-Scholes, with Greeks.
3. **GBM forecasts.** A price fan chart, plus a calibration test of whether GBM's forecast ranges are too narrow given fat tails.
4. **GBM edge signals.** Each option's real-world expected value, using a momentum drift (time-series momentum, Moskowitz et al.) and the HAR forecast volatility, is compared with its implied-vol market price. The gap between implied and forecast volatility is the variance risk premium (Bali et al.).
5. **A DQN trading agent**, set up like the earlier RL notebook. Each day it can **hold, buy a call, buy a put, sell a call, sell a put, or close its oldest option**.
   - Short options post Reg-T style margin and are force-closed on a margin call.
   - A **panic-state filter** blocks *new* short positions after market declines while implied vol is high, following the momentum-crash research (Daniel & Moskowitz).
   - Every trade is tracked: long call, long put, short call and short put. Each records entry/exit, position Greeks, GBM edge and P&L, with statistics by side, type and position.
   - It is benchmarked against a long/short GBM-edge rule, a random policy and buy-and-hold.

**GBM model:** $dS_t = \mu S_t\,dt + \sigma S_t\,dW_t \;\Rightarrow\; S_T = S_0\exp\big((\mu-\tfrac12\sigma^2)T + \sigma\sqrt{T}Z\big)$, with $Z \sim N(0,1)$.
Under the risk-neutral measure $\mu = r$, which gives Black-Scholes. Under the real-world measure, $\mu$ is our drift forecast.

**Backtest hygiene:** Each forecast uses only information available at the close of day *t*. Models are refitted walk-forward each year. Option trades pay a bid/ask spread in vol points. The scaler is fitted on training data only.

> **Data caveat:** free historical option prices aren't available, so options are priced with Black-Scholes at **VIX** as the implied vol. VIX is a 30-day SPX variance-swap rate that includes skew, so it usually sits a little above SPY at-the-money IV. Options are treated as European on adjusted prices, with no dividends or early exercise. Swap in real chain data (e.g. ORATS, OptionMetrics, CBOE DataShop) before trading on these results.

In [1]:
# Imports. Importing func_lib also applies the dark Plotly theme.
import os
import sys

import keras
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.special import expit
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler

# Make func_lib importable whether the notebook runs from this folder or its parent
for path in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), '..'))):
    if os.path.isdir(os.path.join(path, 'quantlib')) and path not in sys.path:
        sys.path.append(path)

from quantlib import func_lib
from quantlib.func_lib import (DOWN, NEUTRAL, OPTION_ACTIONS, PALETTE, RANGE_SELECTOR, TRADING_DAYS, UP,
                      GBM, Agent, bs_greeks, bs_price, build_states, gbm_expected_value, mc_option_price,
                      options_trade_analysis_figure, options_trade_statistics, performance_stats,
                      plot_losses, plot_options_behavior, print_options_summary, run_options_episode,
                      stats_table_figure)

SEED = 42
keras.utils.set_random_seed(SEED)
keras.utils.disable_interactive_logging()


def add_band(fig, x, upper, lower, color, alpha, name=None, **subplot):
    """Shade the area between `upper` and `lower` (two invisible lines, the second filled to the first)."""
    fig.add_scatter(x=x, y=upper, mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip',
                    **subplot)
    fig.add_scatter(x=x, y=lower, mode='lines', line=dict(width=0), fill='tonexty',
                    fillcolor=func_lib._rgba(color, alpha), name=name, showlegend=name is not None,
                    hoverinfo='skip', **subplot)

In [2]:
# ---- Configuration ----
TICKER, IV_TICKER, RF_TICKER = 'SPY', '^VIX', '^IRX'   # underlying, implied-vol proxy, 13-week T-bill
START_DATE, END_DATE = '2007-01-01', None              # None = up to today
IV_FALLBACK_PREMIUM = 0.10    # if IV_TICKER is None: IV = HAR forecast vol * (1 + premium)

FIRST_FORECAST_DATE = '2010-01-01'   # walk-forward forecasts start here (3 years to fit on)
HORIZON = 21                         # trading days: vol-forecast horizon = option tenor (~1 month)
EWMA_LAMBDA = 0.94                   # RiskMetrics
MU_LOOKBACK, MU_SHRINK = 252, 0.5    # GBM drift = 50% trailing 12-month momentum + 50% risk-free rate
FAN_DAYS, N_PATHS = 63, 20_000       # GBM price-forecast horizon and number of simulated paths

# ---- RL options agent (buy AND sell calls / puts) ----
TRAIN_FRACTION = 0.7
WINDOW_SIZE = 1
EPISODES = 3
BATCH_SIZE, TRAIN_EVERY = 32, 1
OPTION_CFG = dict(tenor_days=HORIZON, moneyness=0.0,   # ATM 1-month calls / puts
                  cost_vol=0.01,                       # pay 1 vol point each way (bid/ask)
                  contract_mult=100, capital=100_000,
                  max_positions=6,                     # open contracts in total (long + short)
                  max_short_positions=3,               # cap on naked shorts (tail-risk control)
                  short_margin_pct=0.20, min_margin_pct=0.10)   # Reg-T style naked-option margin
AGENT_PARAMS = dict(action_size=len(OPTION_ACTIONS), memory_size=5000, gamma=0.95, epsilon=1.0,
                    epsilon_min=0.02, epsilon_decay=0.999, learning_rate=1e-3)
EDGE_THRESHOLD = 0.05        # rule benchmark: BUY when the GBM value exceeds the market price by > 5%
SELL_EDGE_THRESHOLD = 0.15   # rule benchmark: SELL when the market price exceeds the GBM value by > 15%
PANIC_FILTER = True          # block NEW shorts in panic states (21d return < 0 and IV above its
PANIC_IV_QUANTILE = 0.80     # training-period 80th percentile)
VERBOSE_TRADES = False

## 1. Data: underlying, implied volatility and risk-free rate

In [3]:
# Download SPY's adjusted OHLCV, plus VIX (implied vol) and the 13-week T-bill yield (risk-free rate).
# VIX and the T-bill are joined onto SPY's dates and forward-filled across holiday mismatches.
ohlc = func_lib.download_ohlcv(TICKER, START_DATE, END_DATE)
aux_tickers = [t for t in (IV_TICKER, RF_TICKER) if t]
aux = func_lib.download_close(aux_tickers, START_DATE, END_DATE)

df = ohlc.join(aux, how='left')
df[aux_tickers] = df[aux_tickers].ffill()
df['rf'] = df[RF_TICKER] / 100                  # annualized decimal
df['log_ret'] = np.log(df['Close']).diff()

required = ['log_ret', 'rf'] + ([IV_TICKER] if IV_TICKER else [])
df = df.dropna(subset=required)

print(f'{TICKER}: {len(df):,} days, {df.index[0]:%Y-%m-%d} to {df.index[-1]:%Y-%m-%d}')

SPY: 4,994 days, 2007-01-04 to 2026-11-09


In [4]:
# SPY's price with VIX underneath. VIX spikes (2008, 2020, 2022) line up with market crashes, and are
# exactly where GBM's constant-volatility assumption is most wrong.
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35], vertical_spacing=0.05,
                    subplot_titles=(f'{TICKER} adjusted close', f'Implied volatility ({IV_TICKER})'))

fig.add_scatter(x=df.index, y=df['Close'], name=TICKER,
                line=dict(color=PALETTE[0], width=1.6), row=1, col=1)
if IV_TICKER:
    fig.add_scatter(x=df.index, y=df[IV_TICKER], name=IV_TICKER,
                    line=dict(color=PALETTE[1], width=1.2), fill='tozeroy',
                    fillcolor=func_lib._rgba(PALETTE[1], 0.10), row=2, col=1)

fig.update_layout(title=f'{TICKER} and Implied Volatility', height=620, showlegend=False,
                  xaxis=dict(rangeselector=RANGE_SELECTOR))
fig.show()

## 2. GBM parameter estimation and its limits
GBM implies normally distributed log returns. The maximum-likelihood estimates are the sample mean and standard deviation of log returns. Equity returns are fat-tailed and negatively skewed, which is why GBM ranges are too narrow in the tails. Section 5 measures how much this matters.

In [5]:
# Fit GBM: σ̂ = std · √252 and μ̂ = mean · 252 + σ̂²/2. Then check the normality assumption.
log_ret = df['log_ret']
gbm_fit = GBM.fit(log_ret, periods_per_year=TRADING_DAYS)
mu_hat, sigma_hat = gbm_fit.mu, gbm_fit.sigma
print(f'GBM MLE (full sample): mu = {mu_hat:.2%} p.a., sigma = {sigma_hat:.2%} p.a.')
print(f'Skewness = {log_ret.skew():.2f} | excess kurtosis = {log_ret.kurt():.2f} (normal = 0)')

# Q-Q plot data: sorted standardized returns vs the matching normal quantiles
z = np.sort(((log_ret - log_ret.mean()) / log_ret.std()).to_numpy())
normal_quantiles = norm.ppf((np.arange(1, len(z) + 1) - 0.5) / len(z))
grid = np.linspace(log_ret.min(), log_ret.max(), 400)

fig = make_subplots(rows=1, cols=2, subplot_titles=('Daily log returns vs fitted normal', 'Normal Q-Q plot'))

# Left: histogram vs fitted normal density
fig.add_histogram(x=log_ret, histnorm='probability density', nbinsx=200, name='Empirical',
                  marker_color=PALETTE[0], opacity=0.75, row=1, col=1)
fig.add_scatter(x=grid, y=norm.pdf(grid, log_ret.mean(), log_ret.std()), name='GBM (normal) fit',
                line=dict(color=PALETTE[1], width=2.5), row=1, col=1)

# Right: Q-Q plot against the 45° line
fig.add_scatter(x=normal_quantiles, y=z, mode='markers', name='Quantiles',
                marker=dict(color=PALETTE[0], size=3), row=1, col=2)
fig.add_scatter(x=[-4, 4], y=[-4, 4], mode='lines', name='45° line',
                line=dict(color=PALETTE[3], dash='dash'), row=1, col=2)

fig.update_xaxes(range=[-0.06, 0.06], title_text='log return', row=1, col=1)
fig.update_xaxes(title_text='Normal quantile', row=1, col=2)
fig.update_yaxes(title_text='Empirical quantile', row=1, col=2)
fig.update_layout(title=func_lib.subtitled('Does GBM fit?', 'Fat tails show up as the Q-Q curve bending away from the line'),
                  height=480, hovermode='closest')
fig.show()

GBM MLE (full sample): mu = 12.17% p.a., sigma = 19.56% p.a.
Skewness = -0.31 | excess kurtosis = 13.96 (normal = 0)


## 3. Volatility forecasting (walk-forward, out-of-sample)
The target is the realized variance over the next 21 trading days. Each model forecasts it at the close of day *t*:

| Model | Inputs |
|---|---|
| Historical 21d | Mean squared return over the last 21 days |
| EWMA | RiskMetrics exponential average, λ = 0.94 |
| GARCH(1,1) | MLE, refitted every January on all prior returns |
| HAR-RV | OLS on daily/weekly/monthly range-based realized variance (overnight gap + Garman-Klass), refitted every January |
| Implied (VIX) | The market's own 30-day forecast; it carries a variance risk premium, so it is biased upward |

QLIKE is the robust loss for variance forecasts: lower is better.

In [6]:
# Each forecaster predicts the mean daily variance over the next HORIZON days, using data up to day t
rv = func_lib.daily_variance_proxy(df).fillna(log_ret ** 2)          # range-based daily realized variance
target = func_lib.forward_realized_variance(log_ret, HORIZON)       # what we are trying to forecast
squared_ret = log_ret ** 2

var_forecasts = {
    'Historical 21d': squared_ret.rolling(21).mean(),
    'EWMA': squared_ret.ewm(alpha=1 - EWMA_LAMBDA, adjust=False).mean(),
    'GARCH(1,1)': func_lib.walk_forward_garch(log_ret, FIRST_FORECAST_DATE, HORIZON),
    'HAR-RV': func_lib.walk_forward_har(rv, target, FIRST_FORECAST_DATE, HORIZON),
}
if IV_TICKER:
    var_forecasts['Implied (VIX)'] = (df[IV_TICKER] / 100) ** 2 / TRADING_DAYS

# Score out-of-sample only
oos = df.index >= pd.Timestamp(FIRST_FORECAST_DATE)
metrics = func_lib.forecast_metrics({name: f[oos] for name, f in var_forecasts.items()}, target[oos])
stats_table_figure(metrics.sort_values('QLIKE'), 'Out-of-sample 21-day volatility forecast accuracy',
                   index_name='Model', decimals=3).show()

In [7]:
# Convert variances to annualized vol, √(variance × 252), and compare forecasts with realized forward vol
vol = {name: np.sqrt(f * TRADING_DAYS) for name, f in var_forecasts.items()}
realized_fwd_vol = np.sqrt(target * TRADING_DAYS)

fig = go.Figure()
fig.add_scatter(x=df.index[oos], y=realized_fwd_vol[oos], name='Realized (next 21d)', mode='lines',
                line=dict(color='#e6e6e6', width=1))
for i, name in enumerate(['HAR-RV', 'GARCH(1,1)', 'Implied (VIX)']):
    if name in vol:
        fig.add_scatter(x=df.index[oos], y=vol[name][oos], name=name, mode='lines',
                        line=dict(color=PALETTE[i], width=1.4))
fig.update_layout(title='21-day volatility: forecasts vs what was realized', height=520,
                  yaxis=dict(title='Annualized vol', tickformat='.0%'),
                  xaxis=dict(rangeselector=RANGE_SELECTOR))
fig.show()

# Variance risk premium (VRP): implied vol minus the vol that was subsequently realized
if IV_TICKER:
    vrp = (vol['Implied (VIX)'] - realized_fwd_vol)[oos].dropna()
    print(f'Implied vol exceeded subsequent realized vol on {(vrp > 0).mean():.0%} of days '
          f'(mean variance risk premium {vrp.mean():.1%} vol points)')

    fig = go.Figure(go.Histogram(x=vrp, nbinsx=120, marker_color=PALETTE[4], opacity=0.8))
    fig.add_vline(x=0, line=dict(color=NEUTRAL, dash='dash'))
    fig.update_layout(title=func_lib.subtitled('Variance risk premium', 'Implied minus subsequently realized 21-day vol'), height=400,
                      xaxis=dict(title='Vol points', tickformat='.0%'), yaxis_title='Days',
                      hovermode='closest')
    fig.show()

Implied vol exceeded subsequent realized vol on 85% of days (mean variance risk premium 3.8% vol points)


## 4. GBM Monte Carlo pricing engine
Monte Carlo under the risk-neutral GBM should converge to Black-Scholes. Antithetic variates (pairing each draw *Z* with −*Z*) cut the standard error for the same number of paths.

In [8]:
# Today's ATM 1-month call: Black-Scholes vs Monte Carlo (plain and antithetic) at growing path counts.
# The 95% bands should close in on the Black-Scholes line.
S0 = df['Close'].iloc[-1]
rf0 = df['rf'].iloc[-1]
sigma_f = vol['HAR-RV'].rename('sigma_f')     # HAR forecast vol, used as GBM's σ from here on
sig0 = sigma_f.iloc[-1]
K0 = S0                                       # at the money
T0 = HORIZON / TRADING_DAYS                   # option tenor in years
bs_call = float(bs_price(S0, K0, T0, rf0, sig0, kind='call'))

path_counts = [500, 1_000, 2_000, 5_000, 10_000, 20_000, 50_000, 100_000, 200_000]
rows = []
for n in path_counts:
    for label, antithetic in [('Plain MC', False), ('Antithetic MC', True)]:
        price, se = mc_option_price(S0, K0, T0, rf0, sig0, kind='call', n_paths=n, seed=SEED,
                                    antithetic=antithetic)
        rows.append({'n': n, 'method': label, 'price': price, 'se': se})
conv = pd.DataFrame(rows)

fig = go.Figure()
for i, (label, g) in enumerate(conv.groupby('method', sort=False)):
    add_band(fig, g['n'], g['price'] + 1.96 * g['se'], g['price'] - 1.96 * g['se'], PALETTE[i], 0.18)
    fig.add_scatter(x=g['n'], y=g['price'], mode='lines+markers', name=f'{label} (95% CI)',
                    line=dict(color=PALETTE[i]))
fig.add_hline(y=bs_call, line=dict(color=PALETTE[3], dash='dash'),
              annotation_text=f'Black-Scholes {bs_call:.3f}', annotation_position='top right')
fig.update_layout(title=func_lib.subtitled(f'MC convergence: ATM {HORIZON}-day call', f'S = {S0:.2f}, σ = {sig0:.1%}'), height=460,
                  xaxis=dict(title='Number of paths', type='log'), yaxis_title='Call price ($)',
                  hovermode='closest')
fig.show()

In [9]:
# Black-Scholes Greeks across spot (85%–115% of S0) for a call and a put. Note delta's S-curve; gamma and
# vega peak at the money and are identical for calls and puts; theta is most negative at the money.
spots = np.linspace(0.85 * S0, 1.15 * S0, 200)
greek_names = {'delta': 'Delta', 'gamma': 'Gamma', 'vega': 'Vega (per vol pt)', 'theta': 'Theta (per day)'}
subplot_pos = [dict(row=i // 2 + 1, col=i % 2 + 1) for i in range(len(greek_names))]   # 2×2 grid

fig = make_subplots(rows=2, cols=2, subplot_titles=list(greek_names.values()),
                    vertical_spacing=0.14, horizontal_spacing=0.08)
for kind, color in [('call', UP), ('put', DOWN)]:
    greeks = bs_greeks(spots, K0, T0, rf0, sig0, kind=kind)
    for i, greek in enumerate(greek_names):
        fig.add_scatter(x=spots, y=greeks[greek], name=kind.title(), legendgroup=kind, showlegend=i == 0,
                        line=dict(color=color, width=2), **subplot_pos[i])
for pos in subplot_pos:
    fig.add_vline(x=K0, line=dict(color=NEUTRAL, dash='dot', width=1), **pos)

fig.update_layout(title=func_lib.subtitled('Black-Scholes Greeks vs spot', f'K = {K0:.0f}, {HORIZON} trading days, σ = {sig0:.1%}'),
                  height=640, hovermode='x unified')
fig.show()

## 5. GBM price forecasts and calibration
**Drift:** $\mu_t$ blends trailing 12-month momentum with the risk-free rate (`MU_SHRINK`). Mean-return estimates are very noisy, so they are shrunk toward the rate.
**Volatility:** the HAR forecast. The fan chart shows the distribution of simulated paths.

In [10]:
# GBM fan chart: drift μ_t blends trailing momentum with the risk-free rate; σ is the HAR forecast
trailing_mu = log_ret.rolling(MU_LOOKBACK).mean() * TRADING_DAYS + 0.5 * sigma_f ** 2
mu_t = (MU_SHRINK * trailing_mu + (1 - MU_SHRINK) * df['rf']).rename('mu')
mu0 = mu_t.iloc[-1]

gbm_today = GBM(mu=mu0, sigma=sig0)
paths = gbm_today.simulate(S0, FAN_DAYS / TRADING_DAYS, FAN_DAYS, N_PATHS, seed=SEED)   # (steps + 1, paths)
future = pd.bdate_range(df.index[-1], periods=FAN_DAYS + 1)
p5, p25, p50, p75, p95 = np.percentile(paths, [5, 25, 50, 75, 95], axis=1)
history = df['Close'].iloc[-126:]                                                       # last ~6 months

fig = go.Figure()
fig.add_scatter(x=history.index, y=history, name='History', line=dict(color='#e6e6e6', width=1.6))
add_band(fig, future, p95, p5, PALETTE[0], 0.12, name='5–95%')
add_band(fig, future, p75, p25, PALETTE[0], 0.25, name='25–75%')
for k in range(30):                                                                     # sample paths
    fig.add_scatter(x=future, y=paths[:, k], mode='lines', showlegend=False, hoverinfo='skip',
                    line=dict(color='rgba(255,255,255,0.10)', width=0.8))
fig.add_scatter(x=future, y=p50, name='Median', line=dict(color=PALETTE[3], width=2.2, dash='dash'))
fig.update_layout(title=func_lib.subtitled(f'GBM {FAN_DAYS}-day forecast for {TICKER}',
                                         f'μ = {mu0:.1%}, σ = {sig0:.1%} ({N_PATHS:,} paths)'), height=540, yaxis_title='Price (USD)')
fig.show()

In [11]:
# Today's ATM 1-month call and put: market price (Black-Scholes at implied vol) vs GBM forecast value
# (real-world drift μ_t and HAR vol), with P(ITM) from simulated terminal prices and Greeks at implied vol.
iv0 = df[IV_TICKER].iloc[-1] / 100 if IV_TICKER else sig0 * (1 + IV_FALLBACK_PREMIUM)
terminal_prices = gbm_today.simulate(S0, T0, HORIZON, N_PATHS, seed=SEED)[-1]

view = {}
for kind in ('call', 'put'):
    market = float(bs_price(S0, K0, T0, rf0, iv0, kind=kind))
    model = float(gbm_today.option_value(S0, K0, T0, rf0, kind))
    p_itm = (terminal_prices > K0).mean() if kind == 'call' else (terminal_prices < K0).mean()
    greeks = bs_greeks(S0, K0, T0, rf0, iv0, kind=kind)
    view[kind.title()] = {
        'Market price (IV)': market,
        'GBM forecast value': model,
        'GBM edge': model / market - 1,
        'P(ITM) under GBM': p_itm,
        'Implied vol': iv0,
        'Forecast vol (HAR)': sig0,
        'Delta': float(greeks['delta']),
        'Vega': float(greeks['vega']),
        'Theta': float(greeks['theta']),
    }

stats_table_figure(pd.DataFrame(view),
                   func_lib.subtitled(f'Today: ATM {HORIZON}-day options', f'{df.index[-1]:%Y-%m-%d}, K = {K0:.2f}'),
                   pct_rows=('GBM edge', 'P(ITM) under GBM', 'Implied vol', 'Forecast vol (HAR)'),
                   decimals=3).show()

**Calibration test.** For every non-overlapping 21-day window since `FIRST_FORECAST_DATE`, we check whether the realized log return fell inside GBM's 50% and 90% ranges, built from each vol forecast. A well-calibrated model hits 50% and 90%. Fat tails show up as too many 5% and 95% exceedances.

In [12]:
# Calibration: standardize each realized 21-day log return by GBM's forecast distribution,
#   z = (log return − (μ − σ²/2)·T) / (σ·√T),
# then count how often z lands inside the 50% / 90% ranges and in each 5% tail. Read this before
# trusting any GBM-based edge.
fwd_log_ret = np.log(df['Close'].shift(-HORIZON) / df['Close'])
windows = df.index[oos][::HORIZON]                 # non-overlapping windows

coverage, pit = {}, {}
for name, sigma in vol.items():
    expected = (mu_t - 0.5 * sigma ** 2) * T0
    z_ret = ((fwd_log_ret - expected) / (sigma * np.sqrt(T0))).reindex(windows).dropna()
    pit[name] = norm.cdf(z_ret)
    coverage[name] = {
        'Inside 50% range': (z_ret.abs() < norm.ppf(0.75)).mean(),
        'Inside 90% range': (z_ret.abs() < norm.ppf(0.95)).mean(),
        'Below 5% quantile': (z_ret < norm.ppf(0.05)).mean(),
        'Above 95% quantile': (z_ret > norm.ppf(0.95)).mean(),
        'Windows': len(z_ret),
    }
coverage = pd.DataFrame(coverage).T
stats_table_figure(coverage, func_lib.subtitled('GBM forecast-range calibration', 'Target: 50% / 90% / 5% / 5%'),
                   index_name='Vol input', pct_rows=tuple(c for c in coverage.columns if c != 'Windows'),
                   decimals=1).show()

# PIT histogram: under perfect calibration each of the 10 bins holds 10% of windows
fig = go.Figure()
for i, name in enumerate(['HAR-RV', 'Implied (VIX)']):
    if name in pit:
        fig.add_histogram(x=pit[name], nbinsx=10, histnorm='probability', name=name,
                          marker_color=PALETTE[i], opacity=0.65)
fig.add_hline(y=0.1, line=dict(color=PALETTE[3], dash='dash'), annotation_text='perfect calibration')
fig.update_layout(title=func_lib.subtitled('Probability integral transform', 'A flat histogram means GBM is calibrated'),
                  barmode='overlay', height=420, xaxis_title='GBM CDF of the realized return',
                  yaxis=dict(title='Share of windows', tickformat='.0%'), hovermode='closest')
fig.show()

## 6. Option features: market price vs GBM forecast value

For the ATM 1-month call and put each day, we compute:
- **market price** = Black-Scholes at implied vol
- **GBM value** = real-world expected payoff with drift $\mu_t$ and HAR vol, discounted at *r*
- **edge** = GBM value / market price − 1

Positive call edge means GBM thinks the call is cheap. Edges are mostly negative because implied vol carries the variance risk premium.

The edge asks one question of each option: does the market charge more or less than GBM expects it to pay? It uses the real-world drift μ and our own volatility forecast, against the market's implied vol. 

In [13]:
# Build `features`: one row per day with prices, vol forecasts, drift, technicals and both option edges.
# bs_price and gbm_expected_value accept whole Series, so each edge is priced for every day in one call.
tech = func_lib.build_ts_features(ohlc)[['SMA_20', 'SMA_50', 'RSI', 'BB_upper', 'BB_lower']]
features = df[['Close', 'rf']].join([sigma_f, mu_t, tech])
features['iv'] = df[IV_TICKER] / 100 if IV_TICKER else features['sigma_f'] * (1 + IV_FALLBACK_PREMIUM)
features['vrp'] = features['iv'] - features['sigma_f']

S, rf = features['Close'], features['rf']          # ATM, so the strike is today's close
for kind in ('call', 'put'):
    market = bs_price(S, S, T0, rf, features['iv'], kind=kind)
    model = gbm_expected_value(S, S, T0, rf, features['mu'], features['sigma_f'], kind)
    features[f'{kind}_edge'] = model / market - 1

features = features[features.index >= pd.Timestamp(FIRST_FORECAST_DATE)].dropna()

fig = go.Figure()
fig.add_scatter(x=features.index, y=features['call_edge'], name='Call edge', line=dict(color=UP, width=1))
fig.add_scatter(x=features.index, y=features['put_edge'], name='Put edge', line=dict(color=DOWN, width=1))
fig.add_hline(y=EDGE_THRESHOLD, line=dict(color=PALETTE[3], dash='dash'),
              annotation_text=f'rule threshold +{EDGE_THRESHOLD:.0%}')
fig.add_hline(y=0, line=dict(color=NEUTRAL, width=1))
fig.update_layout(title=func_lib.subtitled('GBM edge of ATM 1-month options', 'GBM forecast value / market price − 1'),
                  height=480,
                  yaxis=dict(title='Edge', tickformat='.0%'), xaxis=dict(rangeselector=RANGE_SELECTOR))
fig.show()

print(f'Days with call edge > {EDGE_THRESHOLD:.0%}: {(features["call_edge"] > EDGE_THRESHOLD).mean():.1%} | '
      f'put edge > {EDGE_THRESHOLD:.0%}: {(features["put_edge"] > EDGE_THRESHOLD).mean():.1%}')

Days with call edge > 5%: 65.4% | put edge > 5%: 6.4%


## 7. RL options agent (DQN): buy and sell calls and puts

The agent learns a Q-function: for each market-and-book state, the expected discounted future reward of each action. 
It then acts greedily on it. A neural network approximates Q, trained on replayed past experience (`func_lib.Agent`).

It maximizes the discounted return $G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots$ With $\gamma = 0.95$ the effective horizon is about $1/(1-\gamma) = 20$ days, close to the 21-day option life, so the agent weighs a trade's whole lifetime, not just the next day.

The environment (`func_lib.run_options_episode`) turns each daily choice into trades, prices them with spreads, enforces margin and risk limits, marks the book to market and returns the reward. Getting it realistic matters more than the network: an agent learns whatever the environment lets it get away with.


This is the same set-up as the earlier RL notebook, through `func_lib.Agent`, with six actions:

| Action | Meaning | Cash / margin |
|---|---|---|
| 0 | hold | |
| 1 | **buy** 1 ATM 1-month **call** | pay the ask, BS(IV + 1 vol pt) |
| 2 | **buy** 1 ATM 1-month **put** | pay the ask |
| 3 | **sell** 1 ATM 1-month **call** | collect the bid, BS(IV − 1 vol pt), and post margin |
| 4 | **sell** 1 ATM 1-month **put** | collect the bid and post margin |
| 5 | close the oldest open option | longs sell at the bid, shorts buy back at the ask |

**Margin (naked shorts):** premium + max(20%·S − OTM amount, 10%·S) per share, × 100.
- A new trade must fit within *free equity* (equity − margin in use).
- If equity falls below the margin in use, the oldest short is bought back (**margin call**).
- `max_short_positions` caps naked shorts, and the **panic filter** blocks new shorts after a negative month while IV is high.

**State** has three parts:
- sigmoid of the one-day changes and of the levels of the scaled market features;
- the book: share of slots held as long calls, long puts, short calls and short puts;
- average time to expiry and margin utilization.

**Reward** = daily change in account equity as a percent of starting capital. Positions are marked at IV and settle at intrinsic value on expiry.

In [14]:
# Train/test split, panic filter, state construction and the DQN agent
FEATURE_COLS = ['Close', 'SMA_20', 'SMA_50', 'RSI', 'BB_upper', 'BB_lower',
                'sigma_f', 'iv', 'vrp', 'mu', 'call_edge', 'put_edge']

split_date = features.index[int(TRAIN_FRACTION * len(features))]
in_train = features.index < split_date

# Panic state (Daniel & Moskowitz): trailing month negative AND implied vol above its training 80th pct.
# Computed once on the full series, then sliced, so the 21-day return on the first test days uses the
# preceding (training) days instead of being NaN, which would silently switch the filter off there.
panic_iv = features.loc[in_train, 'iv'].quantile(PANIC_IV_QUANTILE)     # threshold from training only
features['panic'] = (features['Close'].pct_change(HORIZON) < 0) & (features['iv'] > panic_iv)
features['short_allowed'] = ~features['panic'] if PANIC_FILTER else True

train_df = features[in_train].copy()
test_df = features[~in_train].copy()
scaler = StandardScaler().fit(train_df[FEATURE_COLS])     # fitted on training data only
print(f'Panic filter: IV threshold {panic_iv:.1%}; new shorts blocked on {features["panic"].mean():.1%} of days'
      + ('' if PANIC_FILTER else ' (filter OFF)'))


def market_states(frame):
    """Sigmoid of the one-day changes and of the levels of the scaled features."""
    z = scaler.transform(frame[FEATURE_COLS])
    return np.hstack([build_states(z, WINDOW_SIZE), expit(z)]).astype(np.float32)


MARKET_DATA_COLS = {'S': 'Close', 'iv': 'iv', 'rf': 'rf', 'sigma_f': 'sigma_f', 'mu': 'mu',
                    'call_edge': 'call_edge', 'put_edge': 'put_edge', 'short_allowed': 'short_allowed'}


def market_data(frame):
    """The arrays run_options_episode needs to price, mark and filter trades."""
    return {'dates': frame.index, **{key: frame[col].to_numpy() for key, col in MARKET_DATA_COLS.items()}}


all_states = market_states(features)       # one pass, so the first test state continues from training
assert np.isfinite(all_states).all()
states_train, states_test = all_states[:len(train_df)], all_states[len(train_df):]
md_train, md_test = market_data(train_df), market_data(test_df)

STATE_SIZE = all_states.shape[1] + func_lib.N_BOOK_FEATURES   # market features + book features
print(f'Train {train_df.index[0]:%Y-%m-%d} to {train_df.index[-1]:%Y-%m-%d} ({len(train_df):,} days) | '
      f'Test {test_df.index[0]:%Y-%m-%d} to {test_df.index[-1]:%Y-%m-%d} ({len(test_df):,} days) | '
      f'state size {STATE_SIZE}')

agent = Agent(state_size=STATE_SIZE, **AGENT_PARAMS)
agent.model.summary()

Panic filter: IV threshold 22.0%; new shorts blocked on 12.7% of days
Train 2010-01-04 to 2021-10-13 (2,966 days) | Test 2021-10-14 to 2026-11-06 (1,272 days) | state size 30


In [15]:
# Training. Each episode replays the training period through the environment, learning as it goes,
# then saves the network so the test cell can reload it.
for e in range(1, EPISODES + 1):
    result = run_options_episode(states_train, md_train, OPTION_CFG, agent=agent, train=True,
                                 batch_size=BATCH_SIZE, train_every=TRAIN_EVERY,
                                 desc=f'Episode {e}/{EPISODES}', verbose=VERBOSE_TRADES)

    print_options_summary(result, f'Training episode {e}/{EPISODES} | epsilon = {agent.epsilon:.3f}')
    plot_options_behavior(result, md_train['S'], f'Training episode {e}')
    plot_losses(result['losses'], f'Episode {e} DQN loss')
    agent.model.save(f'options_dqn_ep{e}.keras')

Episode 1/3: 2,965 steps in 13s

error: operand #1 does not dominate this use
E0000 00:00:1794278586.070418  177478 meta_optimizer.cc:967] tfg_optimizer{any(tfg-consolidate-attrs,tfg-toposort,tfg-shape-inference{graph-version=0},tfg-prepare-attrs-export)} failed: INVALID_ARGUMENT: MLIR Graph Optimizer failed: 
W0000 00:00:1794278586.070559  177478 optimize_function_graph_utils.cc:634] Ignoring multi-device function optimization failure: INVALID_ARGUMENT: MLIR Graph Optimizer failed: 
error: operand #0 does not dominate this use
W0000 00:00:1794278586.134462  177478 optimize_function_graph_utils.cc:634] Ignoring multi-device function optimization failure: INVALID_ARGUMENT: MLIR Graph Optimizer failed: 


----------------------------------------------------------------------
Training episode 1/3 | epsilon = 0.053
Final equity: $88,763.69 (-11.2%) | open at end: 5 | peak margin used: $26,356.56
       All:  797 trades | win rate 46.9% | P&L -$11,236.31 | avg return -3.1%
 Long Call:  303 trades | win rate 48.5% | P&L $7,278.43 | avg return +4.9%
  Long Put:  221 trades | win rate 17.2% | P&L -$39,058.92 | avg return -42.8%
Short Call:   85 trades | win rate 47.1% | P&L -$2,231.38 | avg return -14.4%
 Short Put:  188 trades | win rate 79.3% | P&L $22,775.55 | avg return +36.0%
Max batch loss: 0.2015
----------------------------------------------------------------------


Episode 2/3: 2,965 steps in 12s

----------------------------------------------------------------------
Training episode 2/3 | epsilon = 0.020
Final equity: $87,795.07 (-12.2%) | open at end: 0 | peak margin used: $21,489.49
       All:  637 trades | win rate 41.9% | P&L -$12,204.93 | avg return -5.1%
 Long Call:  310 trades | win rate 42.9% | P&L $9,912.03 | avg return +4.2%
  Long Put:  182 trades | win rate 20.9% | P&L -$33,548.09 | avg return -44.2%
Short Call:   72 trades | win rate 47.2% | P&L -$474.81 | avg return -9.8%
 Short Put:   73 trades | win rate 84.9% | P&L $11,905.95 | avg return +57.8%
Max batch loss: 0.2285
----------------------------------------------------------------------


Episode 3/3: 2,965 steps in 13s

----------------------------------------------------------------------
Training episode 3/3 | epsilon = 0.020
Final equity: $66,719.08 (-33.3%) | open at end: 0 | peak margin used: $19,531.13
       All:  746 trades | win rate 42.5% | P&L -$33,280.92 | avg return -10.5%
 Long Call:  433 trades | win rate 40.6% | P&L -$7,704.86 | avg return -8.1%
  Long Put:  126 trades | win rate 11.1% | P&L -$35,103.41 | avg return -61.5%
Short Call:   87 trades | win rate 52.9% | P&L -$1,602.35 | avg return -3.6%
 Short Put:  100 trades | win rate 81.0% | P&L $11,129.70 | avg return +37.2%
Max batch loss: 0.2170
----------------------------------------------------------------------


## 8. Out-of-sample test vs benchmarks

In [16]:
# Out-of-sample test with exploration off, against two benchmarks run through the same environment
test_agent = Agent(state_size=STATE_SIZE, model_path=f'options_dqn_ep{EPISODES}.keras', test_mode=True,
                   action_size=len(OPTION_ACTIONS))
rl_test = run_options_episode(states_test, md_test, OPTION_CFG, agent=test_agent, desc='Test')


def gbm_edge_rule(t, state):
    """Long/short GBM-edge rule, holding every position to expiry.

    Buy the option GBM rates cheapest when its edge > EDGE_THRESHOLD. Sell the one it rates richest
    when the market price exceeds the GBM value by > SELL_EDGE_THRESHOLD. Otherwise hold.
    """
    call_edge, put_edge = md_test['call_edge'][t], md_test['put_edge'][t]
    scores = {1: call_edge - EDGE_THRESHOLD,         # buy call
              2: put_edge - EDGE_THRESHOLD,          # buy put
              3: -call_edge - SELL_EDGE_THRESHOLD,   # sell call
              4: -put_edge - SELL_EDGE_THRESHOLD}    # sell put
    action, score = max(scores.items(), key=lambda kv: kv[1])
    return action if score > 0 else 0


rng = np.random.default_rng(SEED)


def random_policy(t, state):
    return int(rng.integers(len(OPTION_ACTIONS)))


rule_test = run_options_episode(states_test, md_test, OPTION_CFG, policy=gbm_edge_rule, desc='GBM rule')
random_test = run_options_episode(states_test, md_test, OPTION_CFG, policy=random_policy, desc='Random')

for name, res in [('DQN agent', rl_test), ('GBM-edge rule', rule_test), ('Random policy', random_test)]:
    print_options_summary(res, f'TEST | {name}')
plot_options_behavior(rl_test, md_test['S'], 'Test period | DQN agent')

Test: 1,271 steps in 2s

GBM rule: 1,271 steps in 2s

Random: 1,271 steps in 2s

----------------------------------------------------------------------
TEST | DQN agent
Final equity: $90,693.84 (-9.3%) | open at end: 0 | peak margin used: $0.00
       All:  203 trades | win rate 44.3% | P&L -$9,306.16 | avg return -6.7%
 Long Call:  190 trades | win rate 46.8% | P&L -$2,848.09 | avg return -4.6%
  Long Put:   13 trades | win rate 7.7% | P&L -$6,458.08 | avg return -37.3%
----------------------------------------------------------------------
----------------------------------------------------------------------
TEST | GBM-edge rule
Final equity: $98,758.93 (-1.2%) | open at end: 4 | peak margin used: $54,292.14
       All:  299 trades | win rate 49.5% | P&L -$1,241.07 | avg return +0.3%
 Long Call:  192 trades | win rate 45.3% | P&L -$3,876.80 | avg return +0.1%
  Long Put:   20 trades | win rate 10.0% | P&L -$12,946.99 | avg return -75.2%
Short Call:   24 trades | win rate 50.0% | P&L -$4,948.17 | avg return -17.6%
 Short Put:   63 trades | win rate 74.6% | P&L $20

In [17]:
# Account equity and drawdown for each policy, plus SPY buy-and-hold scaled to the same capital
cap = OPTION_CFG['capital']
curves = {'DQN agent': rl_test['equity'],
          'GBM-edge rule': rule_test['equity'],
          'Random policy': random_test['equity'],
          f'{TICKER} buy & hold': cap * test_df['Close'] / test_df['Close'].iloc[0]}

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3], vertical_spacing=0.06,
                    subplot_titles=('Account equity', 'Drawdown'))
for i, (name, equity) in enumerate(curves.items()):
    color = NEUTRAL if 'buy & hold' in name else PALETTE[i]
    drawdown = equity / equity.cummax() - 1
    fig.add_scatter(x=equity.index, y=equity, name=name, legendgroup=name,
                    line=dict(color=color, width=2), row=1, col=1)
    fig.add_scatter(x=drawdown.index, y=drawdown, name=name, legendgroup=name, showlegend=False,
                    fill='tozeroy', line=dict(color=color, width=1), opacity=0.5, row=2, col=1)
fig.update_yaxes(tickprefix='$', row=1, col=1)
fig.update_yaxes(tickformat='.0%', row=2, col=1)
fig.update_layout(title='Out-of-sample: DQN options agent vs benchmarks', height=680)
fig.show()

perf = pd.DataFrame({name: performance_stats(eq.pct_change().dropna()) for name, eq in curves.items()})
stats_table_figure(perf, 'Out-of-sample performance (account level)',
                   pct_rows=('Total Return', 'CAGR', 'Annualized Volatility', 'Max Drawdown', 'Positive Days')).show()

## 9. Trade tracking: long and short calls and puts
Return % is relative to the premium paid for longs and to the premium collected for shorts. Greeks are *position* Greeks, so a short call has negative delta and positive theta.

In [18]:
# Trade statistics grouped by All, Side (long/short), Type (call/put) and Position (the four combinations)
trades = rl_test['trades']
PCT_ROWS = ('Win Rate', 'Held to Expiry', 'Avg Entry IV')

for name, res in [('DQN agent', rl_test), ('GBM-edge rule', rule_test)]:
    stats_table_figure(options_trade_statistics(res['trades']),
                       func_lib.subtitled(f'{name}: trade statistics (test)', 'By side, type and position'),
                       index_name='Statistic', pct_rows=PCT_ROWS).show()

In [19]:
# Trade log: every long/short call and put with entry/exit, position Greeks at entry, GBM edge and P&L
TRADE_COLS = ['Position', 'Side', 'Type', 'Entry Date', 'Exit Date', 'Exit Reason', 'Strike', 'Entry Spot',
              'Exit Spot', 'Entry IV', 'Forecast Vol', 'GBM Edge %', 'Entry Premium', 'Exit Value', 'P&L ($)',
              'Return %', 'Holding Days', 'Delta', 'Gamma', 'Vega', 'Theta']

trade_log = trades[TRADE_COLS].copy() if len(trades) else pd.DataFrame(columns=TRADE_COLS)
trade_log.to_csv('options_agent_trade_log.csv', index=False)
print(f'{len(trade_log):,} trades saved to options_agent_trade_log.csv')

numeric_cols = trade_log.select_dtypes('number').columns
trade_log.tail(10).round({c: 3 for c in numeric_cols})

203 trades saved to options_agent_trade_log.csv


,Position,Side,Type,Entry Date,Exit Date,Exit Reason,Strike,Entry Spot,Exit Spot,Entry IV,...,GBM Edge %,Entry Premium,Exit Value,P&L ($),Return %,Holding Days,Delta,Gamma,Vega,Theta
193,Long Call,Long,Call,2026-05-20,2026-06-04,Agent close,737.514,737.514,753.274,0.174,...,11.977,16.753,19.837,308.377,18.408,10,0.533,0.011,0.846,-0.279
194,Long Call,Long,Call,2026-05-22,2026-06-24,Expired,741.882,741.882,731.424,0.167,...,19.137,16.231,0.000,-1623.076,-100.000,21,0.534,0.011,0.851,-0.271
195,Long Call,Long,Call,2026-05-26,2026-06-25,Expired,746.807,746.807,732.481,0.170,...,17.702,16.603,0.000,-1660.324,-100.000,21,0.534,0.011,0.857,-0.277
196,Long Call,Long,Call,2026-06-01,2026-07-01,Expired,754.717,754.717,743.913,0.160,...,19.489,15.960,0.000,-1595.991,-100.000,21,0.535,0.011,0.866,-0.267
197,Long Put,Long,Put,2026-06-02,2026-07-02,Expired,755.742,755.742,742.935,0.158,...,-36.523,13.463,12.806,-65.662,-4.877,21,-0.465,0.012,0.867,-0.189
198,Long Call,Long,Call,2026-06-10,2026-07-13,Expired,721.774,721.774,747.314,0.222,...,1.568,20.380,25.541,516.112,25.325,21,0.532,0.009,0.829,-0.339
199,Long Call,Long,Call,2026-06-16,2026-07-17,Expired,746.548,746.548,741.449,0.164,...,29.309,16.099,0.000,-1609.875,-100.000,21,0.535,0.011,0.856,-0.269
200,Long Call,Long,Call,2026-07-06,2026-08-04,Expired,749.419,749.419,769.419,0.156,...,33.021,15.458,20.000,454.266,29.388,21,0.536,0.012,0.860,-0.259
201,Long Call,Long,Call,2026-07-13,2026-08-11,Expired,747.314,747.314,768.651,0.172,...,13.190,16.790,21.337,454.739,27.084,21,0.535,0.011,0.857,-0.281
202,Long Call,Long,Call,2026-08-13,2026-09-14,Expired,775.953,775.953,758.995,0.146,...,24.392,15.173,0.000,-1517.330,-100.000,21,0.538,0.012,0.890,-0.255


In [20]:
# Dashboards: trade analysis, monthly account-return heatmap, and the actions the agent chose
if len(trades):
    options_trade_analysis_figure(trades, func_lib.subtitled('DQN agent: trade analysis (test)',
                                                             'Long & short calls and puts')).show()

    # Monthly returns as a Year × Month grid (the first month is measured from starting capital)
    month_end = rl_test['equity'].resample('ME').last()
    monthly = month_end.pct_change()
    monthly.iloc[0] = month_end.iloc[0] / cap - 1
    grid = (monthly.to_frame('ret')
            .assign(Year=monthly.index.year, Month=monthly.index.strftime('%b'))
            .pivot(index='Year', columns='Month', values='ret'))
    month_order = pd.date_range('2000-01-01', periods=12, freq='MS').strftime('%b')
    grid = grid[[m for m in month_order if m in grid]]

    values = grid.to_numpy()
    lim = np.nanmax(np.abs(values))                  # symmetric color scale around 0
    labels = np.vectorize(lambda v: '' if np.isnan(v) else f'{v:.1%}')(values)
    fig = go.Figure(go.Heatmap(z=values, x=grid.columns, y=grid.index.astype(str),
                               zmid=0, zmin=-lim, zmax=lim, colorscale=[[0, DOWN], [0.5, '#1b1f2a'], [1, UP]],
                               text=labels, texttemplate='%{text}',
                               hovertemplate='%{y} %{x}: %{z:.2%}<extra></extra>',
                               colorbar=dict(tickformat='.1%')))
    fig.update_layout(title='DQN agent: monthly account returns (test)', height=120 + 45 * len(grid),
                      hovermode='closest')
    fig.show()

ACTION_COLORS = {'buy call': UP, 'buy put': DOWN, 'sell call': PALETTE[5], 'sell put': PALETTE[0]}
action_counts = pd.Series(rl_test['actions'][:-1]).map(OPTION_ACTIONS).value_counts()
fig = go.Figure(go.Bar(x=action_counts.index, y=action_counts.values,
                       marker_color=[ACTION_COLORS.get(a, NEUTRAL) for a in action_counts.index]))
fig.update_layout(title='DQN agent: actions chosen in the test period', height=380, yaxis_title='Days',
                  hovermode='closest')
fig.show()

## 10. Today's signal from the trained agent (assumes an empty book)

In [21]:
# Today's state with an empty book. Mask actions that aren't available, then take the best Q-value.
latest_state = np.concatenate([all_states[-1], np.zeros(func_lib.N_BOOK_FEATURES, dtype=np.float32)])
q_values = test_agent.q_values(latest_state)[0]
actions = range(len(q_values))

CLOSE, SELL_CALL, SELL_PUT = 5, 3, 4
unavailable = {CLOSE: 'no (empty book)'}
if not features['short_allowed'].iloc[-1]:
    unavailable[SELL_CALL] = unavailable[SELL_PUT] = 'no (panic filter)'
best_action = max((a for a in actions if a not in unavailable), key=lambda a: q_values[a])

signal = pd.DataFrame({'Q-value': q_values, 'Available': [unavailable.get(a, 'yes') for a in actions]},
                      index=[OPTION_ACTIONS[a] for a in actions])

today = features.iloc[-1]
print(f'{features.index[-1]:%Y-%m-%d} | spot {today["Close"]:.2f} | IV {today["iv"]:.1%} | '
      f'HAR vol {today["sigma_f"]:.1%} | call edge {today["call_edge"]:+.1%} | put edge {today["put_edge"]:+.1%}')
print(f'Agent action: {OPTION_ACTIONS[best_action].upper()}')
stats_table_figure(signal, 'Agent Q-values for today', index_name='Action', decimals=4).show()

2026-11-06 | spot 735.29 | IV 25.6% | HAR vol 17.4% | call edge -25.2% | put edge -38.4%
Agent action: HOLD


## 11. Takeaways and next steps
- **Volatility forecasts.** Expect HAR-RV and GARCH to beat historical vol on QLIKE. VIX usually has the highest R² but a positive bias: that bias *is* the variance risk premium.
- **GBM calibration.** The 90% range typically catches fewer than 90% of outcomes, with too many crash-side exceedances. GBM understates tail risk, so treat its option values as a baseline, not the truth.
- **Edges.** Options usually look expensive versus GBM because implied vol exceeds realized vol. That is why **selling** is valuable: short puts and calls harvest the variance risk premium. The cost is negative skew, with small frequent gains and rare large losses.
- **Risk controls matter more than the signal for shorts.** Compare the margin-call count, worst trades and drawdown with the panic filter on and off (`PANIC_FILTER`), and with different `max_short_positions`.
- **Next steps**
  - Add defined-risk structures (put spreads, iron condors) so short premium has a capped loss.
  - Size positions on vega or margin rather than in fixed contracts.
  - Replace GBM with **Heston or rough Bergomi** volatility.
  - Use **real option chains** with skew.
  - Add RFSV as a forecaster.
  - Train on several underlyings to reduce overfitting.